In [44]:
import numpy as np
import json
# with open(r'C:\Users\juani\Documents\Github\Abaqus_WELL_\wellClosure_axi.json') as f:
with open(r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL\wellbore_closure_planestrain.json') as f:
    data = json.load(f)

def process_lithology(data, global_depth):

    # filtered_rocks = {}
    # filtered_layers = []
    l_depth = data["AnalysisData"]["Depth"]
    lithology = data["Lithology"]
    json_rocks = data["Rocks"]

    for layer in lithology:
        l_top = layer["Top"] 
        l_bottom = layer["Bottom"]
        
        if l_bottom <= l_depth and l_top <= l_depth:
            print(l_top, l_bottom)
        else:
            continue
            
        # Formata o nome da camada
        # Prepara o material da rocha
        rock_name = layer["Rock"]
        rock_mat = json_rocks[rock_name].copy()
        mat_name = "LAYER_%s" % rock_name # Ex: LAYER_SANDSTONE
        print(rock_mat.keys())
        rock_mat["Name"] = mat_name  
        # filtered_rocks[mat_name] = rock_mat
       
        print(f"Processed layer: {mat_name} with top at {l_depth} meters") 
        print(f"Original layer: {rock_name} at {l_depth} meters")
        # filtered_layers.append(new_layer)
    
    return global_depth, l_depth

global_depth = data["AnalysisData"]["Depth"]
print(f"Global depth for analysis: {global_depth} meters")
global_depth, l_depth = process_lithology(data, global_depth)

print("Finished processing lithology.")

Global depth for analysis: 2500.0 meters
2000.0 2500.0
dict_keys(['MohrCoulombParameters', 'Law', 'Elasticity', 'ElasticParameters', 'ThermalParameters'])
Processed layer: LAYER_SHALE with top at 2500.0 meters
Original layer: SHALE at 2500.0 meters
Finished processing lithology.


In [52]:
import os
import json
import sys

path_project = r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL'

# Reading the json file and filling the input data for the analysis ####################
with open(r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL\wellbore_closure_planestrain.json') as f:
    data = json.load(f)
    # print(f"Data keys: {data.keys()}")
# Data keys: dict_keys(['AnalysisData', 'ThermalGradient',
#           'Tubulars', 'Lithology', 'InSituStresses', 'Rocks', 'Cements',
#           'SteelGrades', 'Phases', 'Events', 'Fluids'])

# Variables read from json (geometry) #####################################

# name_phase = '3dda7930-6dbf-4d05-87f2-d2809a3e9fc6'
if "Phases" in data["AnalysisData"]:
    name_phase = data["AnalysisData"]["Phases"]
    print(name_phase)
phase_data = data["Phases"][name_phase]
print(phase_data)
if phase_data:
    name_tubular = phase_data["Casing"][0]["Tubular"]
    print(f"Tubular used in phase '{name_phase}': {name_tubular}")
else:
    print(f"Phase '{name_phase}' not found in data['Phases']")

diameter_wellbore = phase_data["HoleDiameter"]
print(diameter_wellbore)

812c492a-c945-4184-be51-841c5fb86b15
{'Casing': [{'Tubular': 'VAM_20_#169_K55', 'Top': 2000.0, 'Bottom': 3200.0}], 'CementSheath': [{'Cement': 'CLASS_G', 'Top': 2500.0, 'Bottom': 3200.0}], 'HoleDiameter': 26.0, 'Standoff': 100.0, 'RateOfPenetration': 10.0, 'TripSpeed': 400.0, 'Fluid': 'drilling 20 in', 'Name': 'surface'}
Tubular used in phase '812c492a-c945-4184-be51-841c5fb86b15': VAM_20_#169_K55
26.0


In [49]:
import os
import json
import sys

path_project = r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL'

# Reading the json file and filling the input data for the analysis ####################
with open(r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL\wellbore_closure_planestrain.json') as f:
    data = json.load(f) 

l_depth = data["AnalysisData"]["Depth"]
lithology = data["Lithology"]
print(l_depth)

for item in lithology:
    if l_depth >= item["Top"] and l_depth < item["Bottom"]:
        layer_rock = item["Rock"]
        print(f"Layer at depth {l_depth} meters: {layer_rock}")

casing_type = data["Tubulars"][name_tubular]["Material"] 
steelgrade_info = data["SteelGrades"][casing_type]
data_geothermal = data["AnalysisData"]["GeothermalGradient"]
thermalGradient = data["ThermalGradient"][data_geothermal]

examples = {}

name_fluido = next(
        (name for name, info in data["Fluids"].items() 
         if info.get("ThermalGradient") == data_geothermal), None)
print(f"Selected Fluid: {name_fluido}")

examples["STEEL"] = {
        "behavior": data["SteelGrades"][casing_type]["Law"],
        'density': data["SteelGrades"][casing_type]["ElasticParameters"]["Density"],
        'elastic': (data["SteelGrades"][casing_type]["ElasticParameters"]["Young"]*1e9,
                    data["SteelGrades"][casing_type]["ElasticParameters"]["Poisson"]),
        'conductivity': data["SteelGrades"][casing_type]["ThermalParameters"]["Conductivity"],
        'specific_heat': data["SteelGrades"][casing_type]["ThermalParameters"]["SpecificHeat"],
        'expansion': data["SteelGrades"][casing_type]["ThermalParameters"]["ThermalExpansion"],
        "type": "Casing"
    }

examples["FLUID"] = {
        "behavior": "ELASTIC",
        'density': data["Fluids"][name_fluido]["Density"],
        'compressibility': data["Fluids"][name_fluido]["Compressibility"],
        'ThermalExpansion': data["Fluids"][name_fluido]["ThermalExpansion"],
        "type": "Fluid"
    }

############# PAREI AQUI #############################

examples[layer_rock] = {
    "behavior": data["Rocks"][layer_rock]["Law"],
    'density': data["Rocks"][layer_rock]["ElasticParameters"]["Density"],
    'elastic': (data["Rocks"][layer_rock]["ElasticParameters"]["Young"]*1e9,
                data["Rocks"][layer_rock]["ElasticParameters"]["Poisson"]),
    'conductivity': data["Rocks"][layer_rock]["ThermalParameters"]["Conductivity"],
    'specific_heat': data["Rocks"][layer_rock]["ThermalParameters"]["SpecificHeat"],
    'expansion': data["Rocks"][layer_rock]["ThermalParameters"]["ThermalExpansion"],
    "type": "Rock"
}


if "MohrCoulombParameters" in data["Rocks"][layer_rock]:
    mc = data["Rocks"][layer_rock]["MohrCoulombParameters"]
    examples[layer_rock].update({
        'friction_angle': mc["FrictionAngle"],
        'dilatancy_angle': mc["DilatancyAngle"],
        'cohesion': mc["Cohesion"],
        "lab_data": ((20001698.76, 0.0), )
    })

print(f"Examples dictionary: {examples}", sep="\n")
# for layer_rock, properties in layer_rock.items():

#     examples[layer_rock] = {
#         "behavior": properties["Law"],
#         'density': properties["ElasticParameters"]["Density"],
#         'elastic': (properties["ElasticParameters"]["Young"]*1e9,
#                     properties["ElasticParameters"]["Poisson"]),
#         'conductivity': properties["ThermalParameters"]["Conductivity"],
#         'specific_heat': properties["ThermalParameters"]["SpecificHeat"],
#         'expansion': properties["ThermalParameters"]["ThermalExpansion"],
#         "type": "Rock"
#     }

#     if "MohrCoulombParameters" in properties:
#         mc = properties["MohrCoulombParameters"]
#         examples[layer_rock].update({
#             'friction_angle': mc["FrictionAngle"],
#             'dilatancy_angle': mc["DilatancyAngle"],
#             'cohesion': mc["Cohesion"],
#             "lab_data": ((20001698.76, 0.0), )
#         })

2600.0
Layer at depth 2600.0 meters: SANDSTONE
Selected Fluid: drilling_14in (60 deg)
Examples dictionary: {'STEEL': {'behavior': 'VON_MISES_PLASTIC', 'density': 7935.0, 'elastic': (206842800000.0, 0.3), 'conductivity': 45.3452, 'specific_heat': 460.9, 'expansion': 0.0001242, 'type': 'Casing'}, 'FLUID': {'behavior': 'ELASTIC', 'density': 10.5, 'compressibility': 1e-05, 'ThermalExpansion': 0.0001242, 'type': 'Fluid'}, 'SANDSTONE': {'behavior': 'MOHR_COULOMB', 'density': 2300.0, 'elastic': (24062023000.0, 0.25), 'conductivity': 1.869, 'specific_heat': 0.209946, 'expansion': 1e-05, 'type': 'Rock', 'friction_angle': 30.0, 'dilatancy_angle': 7.5, 'cohesion': 20.0017, 'lab_data': ((20001698.76, 0.0),)}}
